# Question 1

Vectorized operations on sensor readings.

In [1]:
import numpy as np

temperature = np.array([25, 28, 31, 35, 38, 27, 33, 40])
temperature

array([25, 28, 31, 35, 38, 27, 33, 40])

### a. Add 2°C to every reading

In [2]:
corrected = temperature + 2
corrected

array([27, 30, 33, 37, 40, 29, 35, 42])

### b. Convert all temperatures from Celsius to Fahrenheit

In [3]:
fahrenheit = temperature * 9 / 5 + 32
fahrenheit

array([ 77. ,  82.4,  87.8,  95. , 100.4,  80.6,  91.4, 104. ])

### c. Readings greater than 32°C

In [4]:
above_32 = temperature[temperature > 32]
above_32

array([35, 38, 33, 40])

### d. Count of readings above 32°C

In [5]:
count_above_32 = np.sum(temperature > 32)
print("Readings above 32°C:", count_above_32)

Readings above 32°C: 4


### e. Why vectorization and Boolean indexing are more efficient

- Vectorized operations run in compiled C loops inside NumPy, so the per-element Python interpreter overhead of a `for` loop is avoided.
- The array is stored in one contiguous block of a single data type, which is cache friendly and lets the CPU use SIMD instructions.
- Boolean indexing builds a mask (`temperature > 32`) in one pass and selects the matching values in a second pass, with no `if` or `append` for each value.
- The code is shorter and easier to read: `temperature * 9 / 5 + 32` states the formula directly.

# Question 2

Daily steps recorded by four users over three days.

In [6]:
steps = np.array([
    [5000, 6200, 7100],
    [8000, 7500, 9000],
    [4500, 5100, 4800],
    [9000, 8500, 9500],
])
steps

array([[5000, 6200, 7100],
       [8000, 7500, 9000],
       [4500, 5100, 4800],
       [9000, 8500, 9500]])

### a. Total steps recorded

In [7]:
np.sum(steps)

np.int64(84200)

### b. Mean number of steps

In [8]:
np.mean(steps)

np.float64(7016.666666666667)

### c. Maximum and minimum values

In [9]:
print("Maximum:", np.max(steps))
print("Minimum:", np.min(steps))

Maximum: 9500
Minimum: 4500


### d. Total steps for each day (`axis=0`)

In [10]:
daily_total = np.sum(steps, axis=0)
daily_total

array([26500, 27300, 30400])

### e. Total steps for each user (`axis=1`)

In [11]:
user_total = np.sum(steps, axis=1)
user_total

array([18300, 24500, 14400, 27000])

### f. Position of the maximum number of steps

In [12]:
flat_idx = np.argmax(steps)
user, day = (int(i) for i in np.unravel_index(flat_idx, steps.shape))

print("Flat index :", flat_idx)
print("User, Day  :", (user, day), "(0-indexed)")
print("Steps      :", steps[user, day])

Flat index : 11
User, Day  : (3, 2) (0-indexed)
Steps      : 9500


# Question 3

Views, copies, indexing and flattening.

### a. Create the array

In [13]:
original = np.array([1, 2, 3, 4, 5, 6])
original

array([1, 2, 3, 4, 5, 6])

### b. Slice from index 1 to 4

In [14]:
subset = original[1:5]
subset

array([2, 3, 4, 5])

### c. Modify the first element of `subset`

In [15]:
subset[0] = 999

print("original:", original)
print("subset  :", subset)

original: [  1 999   3   4   5   6]
subset  : [999   3   4   5]


A basic slice is a view, so changing `subset` also changed `original`.

### d. Slice with `.copy()`

In [16]:
copied = original[1:5].copy()
copied[0] = 500

print("original:", original)
print("copied  :", copied)

original: [  1 999   3   4   5   6]
copied  : [500   3   4   5]


The copy owns its own data, so `original` is unchanged.

### e. Numbers 1 to 12 as a 3 × 4 matrix

In [17]:
matrix = np.arange(1, 13).reshape(3, 4)
matrix

array([[ 1,  2,  3,  4],
       [ 5,  6,  7,  8],
       [ 9, 10, 11, 12]])

### f. Indexing and slicing

In [18]:
print("First row      :", matrix[0])
print("Last row       :", matrix[-1])
print("Second column  :", matrix[:, 1])
print("Rows 1-2, cols 2-3:")
print(matrix[1:3, 2:4])

First row      : [1 2 3 4]
Last row       : [ 9 10 11 12]
Second column  : [ 2  6 10]
Rows 1-2, cols 2-3:
[[ 7  8]
 [11 12]]


### g. Flatten with `flatten()` and `ravel()`

In [19]:
flattened = matrix.flatten()
raveled = matrix.ravel()

print("flatten():", flattened)
print("ravel()  :", raveled)

flatten(): [ 1  2  3  4  5  6  7  8  9 10 11 12]
ravel()  : [ 1  2  3  4  5  6  7  8  9 10 11 12]


### h. Modify the array returned by `ravel()`

In [20]:
raveled[0] = 100

print("raveled :", raveled)
print("matrix  :")
print(matrix)

raveled : [100   2   3   4   5   6   7   8   9  10  11  12]
matrix  :
[[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]


`ravel()` returns a view when possible, so the original matrix changed.

### i. Modify the array returned by `flatten()`

In [21]:
flattened[1] = 200

print("flattened:", flattened)
print("matrix   :")
print(matrix)

flattened: [  1 200   3   4   5   6   7   8   9  10  11  12]
matrix   :
[[100   2   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]


`flatten()` always returns a copy, so the original matrix is unaffected.

### j. Attributes of the 3 × 4 matrix

In [22]:
print("Shape:", matrix.shape)
print("ndim :", matrix.ndim)
print("size :", matrix.size)
print("dtype:", matrix.dtype)

Shape: (3, 4)
ndim : 2
size : 12
dtype: int64


# Question 4

Predicting an assistance score with Ordinary Least Squares.

In [23]:
X = np.array([
    [6, 70, 3],
    [5, 50, 6],
    [8, 80, 2],
    [4, 30, 8],
])

y = np.array([40, 65, 30, 85])

### a. Shape and dimensions of X

In [24]:
print("Shape:", X.shape)
print("ndim :", X.ndim)

Shape: (4, 3)
ndim : 2


X has 4 rows (users) and 3 columns (sleep hours, activity level, stress level).

### b. Transpose of X

In [25]:
X.T

array([[ 6,  5,  8,  4],
       [70, 50, 80, 30],
       [ 3,  6,  2,  8]])

The transpose swaps rows and columns. `X.T` has shape (3, 4): each row now holds one feature across all four users.

### c. Matrix product `X.T @ X`

In [26]:
XtX = X.T @ X
XtX

array([[  141,  1430,    96],
       [ 1430, 14700,   910],
       [   96,   910,   113]])

This is a 3 × 3 matrix. Entry (i, j) is the dot product of feature i and feature j across all users, so it captures how the features relate to each other.

### d. Matrix inverse

In [27]:
print("Determinant:", np.linalg.det(XtX))
print("Rank       :", np.linalg.matrix_rank(XtX))

XtX_inv = np.linalg.inv(XtX)
XtX_inv

Determinant: 753700.000000011
Rank       : 3


array([[ 1.10521428, -0.09848746, -0.14581398],
       [-0.09848746,  0.00891203,  0.01190129],
       [-0.14581398,  0.01190129,  0.0368847 ]])

The inverse only exists when the determinant is non-zero (full rank). Here `X.T @ X` is invertible, so `np.linalg.inv()` can be used.

In [28]:
np.round(XtX @ XtX_inv, 6)

array([[ 1.,  0., -0.],
       [ 0.,  1., -0.],
       [-0.,  0.,  1.]])

### e. Ordinary Least Squares: β = (XᵀX)⁻¹ Xᵀ y

In [29]:
beta = XtX_inv @ X.T @ y
beta

array([-0.15921454,  0.14083853, 10.06302242])

In [30]:
np.allclose(beta, np.linalg.lstsq(X, y, rcond=None)[0])

True

### f. Meaning of the coefficients

Each coefficient is the change in predicted assistance score for a one-unit increase in that feature, keeping the other two fixed:

- **Sleep hours** (β₀ ≈ -0.16): one more hour of sleep lowers the score slightly.
- **Activity level** (β₁ ≈ 0.14): one more unit of activity raises the score slightly.
- **Stress level** (β₂ ≈ 10.06): one more unit of stress raises the score by about 10 points, so stress is by far the dominant factor.

A negative value lowers the predicted score and a positive value raises it. The model has no intercept term, so the prediction is purely a weighted sum of the three features. With only 4 users and 3 features, the fit is very sensitive to the data, so the coefficients should not be over-interpreted.

### g. Prediction for a new user

In [31]:
new_user = np.array([5, 40, 7])
predicted_score = new_user @ beta
print("Predicted assistance score:", predicted_score)

Predicted assistance score: 75.27862544779117
